**Vision-Based Gaze Estimation | Presented by: Reza Saadatyar (2026)**

**04 | CNN and Transfer-Learning Gaze Estimation**

![Neural gaze-estimation workflow and recorded two-fold development evidence](../../images/04_cnn_and_transfer_learning.svg)

**Objective:** Develop a compact grayscale CNN and ImageNet-initialised ResNet-18 using the same 7,500 eye images and participant roles as the classical models.

**Recorded development results:** On the first two of 15 outer-fold configurations, the mean of the selected inner-Validation errors was 6.0758° for Small CNN and 8.0542° for ResNet-18. The remaining 13 configurations were not evaluated in that development run. The current notebook enables complete outer LOPO with the same model procedures; final results are not yet available.

**Scope:** This two-fold experiment verifies neural training and checkpoint selection on the study-defined MPIIGaze subset. It does not establish full LOPO generalisation.

**00. Colab Repository Setup**

In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess
repo_url="https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path=Path("/content")/repo_url.rsplit("/",1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git","clone","-q",repo_url,str(repo_path)],check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}

/content/Eye-Tracking-Neurocognitive-Intelligence


**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root=setup_colab("vision_based_gaze_estimation")

# ---------------------------- 2. Standard Library ----------------------------
import random
import sys

# ----------------- 3. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 4. Machine Learning and Deep Learning -------------------
import torch
from gaze_models import train_gaze_network, predict_network
from gaze_geometry import angular_error_degrees
from mpiigaze_data import restore_indexed_eye_cache
from gaze_evaluation import (load_image_cache, outer_fold_indices,
                             validate_lopo_design,
                             reconcile_outer_predictions)

# ------------------- 5. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure

# ------------------ 6. Shared and Study Research Utilities -------------------
from core.computer_device import computer_device
from core.output_download import download_outputs_from_colab

# ------------------- 7. Reproducibility and Compute Device -------------------
SEED=17
random.seed(SEED)
np.random.seed(SEED)
device=computer_device(min_vram_gb=None)

# ----------------------- 8. Output Directory and Setup -----------------------
out_dir=root/"vision_based_gaze_estimation"/"output"
cache_dir=root/"data"/"processed"/"mpiigaze"
cache_path=cache_dir/"02_normalised_eye_cache.npz"
checkpoint_dir=cache_dir/"checkpoints"
checkpoint_dir.mkdir(parents=True,exist_ok=True)
plot_style(font_family="DejaVu Sans",title_size=12.5,
 title_weight="normal",label_size=11.5,label_weight="normal",
 tick_size=11,legend_size=10,figure_dpi=100,figure_size=(9,4))
print("Study outputs:",out_dir)
print("Local-only model checkpoints:",checkpoint_dir)
print("Python:",sys.version.split()[0])

**02. Check Original Images and Fold Identities**

The recorded run loaded all 7,500 source-linked images from 15 participants and the 15 fixed outer folds. The cache loader reconciles sample order and 3D targets with the Notebook 02 index. Missing local images can be restored from the exact indexed source rows without resampling or changing the folds.

In [ ]:
# ================= 02. Verify Current Image and Fold Inputs ==================

required=["02_sample_index.csv","02_fold_assignments.csv"]
missing=[x for x in required if not (out_dir/x).is_file()]
if missing:
    raise FileNotFoundError(f"Notebook 02 outputs missing: {missing}")
# Recover the exact Stage-02 sample rows if this runtime is new.
if not cache_path.is_file():
    restore_indexed_eye_cache(
        root / "data" / "raw" / "mpiigaze",
        out_dir / "02_sample_index.csv", cache_path)
index,images,targets=load_image_cache(
 out_dir/"02_sample_index.csv",cache_path)
folds=pd.read_csv(out_dir/"02_fold_assignments.csv",
 dtype={"participant_id":str,"fold_id":str,"outer_test_person":str})
if folds.duplicated(["fold_id","participant_id"]).any() or folds.empty:
    raise ValueError("Outer-fold source identities are repeated or missing")
validate_lopo_design(index, folds)
print("Matching original image examples:",len(index))
print("Independent people:",index.participant_id.nunique())
print("Available outer folds:",folds.fold_id.nunique())

**03. Select Development Mode and Model Configurations**

The earlier development run used outer_01 and outer_02. The current final mode evaluates all 15 configurations. Each has 6,000 Training images from 12 people, 1,000 inner Validation images from two people and 500 reserved Test images from one person.

Both networks use AdamW, cosine-direction loss, batch size 128, eight maximum epochs and patience three. The learning rate is 0.001 for Small CNN and 0.0001 for ImageNet-initialised ResNet-18. The base seed is 17; the seed increases by one per outer fold (17–31 in the final run). A complete outer-LOPO run uses all 15 configurations under the fixed procedures.

In [ ]:
# =================== 03. Configure Nested Neural Training ====================

# ------------------- 1. Explicit Compute and Test Settings -------------------
RUN_OUTER_TEST=True
MAX_OUTER_FOLDS=2 if not RUN_OUTER_TEST else None
MODEL_NAMES=("small_cnn","resnet18")
PRETRAINED_RESNET=True
EPOCHS=8
PATIENCE=3
BATCH_SIZE=128
LEARNING_RATE=1e-3

# ------------------------ 2. Preserve Test Boundaries ------------------------
all_folds=sorted(folds.fold_id.unique())
active_folds=(all_folds if MAX_OUTER_FOLDS is None
              else all_folds[:MAX_OUTER_FOLDS])
if RUN_OUTER_TEST and len(active_folds)!=len(all_folds):
    raise ValueError("A final outer LOPO run requires all outer participants")
if not RUN_OUTER_TEST:
    prior=[f for f in ("04_neural_outer_predictions.csv",
                       "04_neural_outer_metrics.csv") if (out_dir/f).exists()]
    if prior:
        raise RuntimeError("Older Test files exist; preserve the current evidence "
                           "and use a clean development checkout: "+str(prior))
print("Current mode:","outer Test" if RUN_OUTER_TEST else "inner Validation development")
print("Outer fold configurations:",len(active_folds),"/",len(all_folds))
print("Model families:",MODEL_NAMES)

**04. Train Small CNN and ResNet-18 Within Fold Roles**

The Small CNN has three convolutional blocks and 48,387 trainable parameters. ResNet-18 adapts the first convolution to grayscale by averaging pretrained RGB filters and replaces its classification head with a three-component gaze head (11,171,779 parameters). Both models optimise cosine-direction loss on Training samples and select the checkpoint with the smallest inner-Validation angular error.

| Fold | Small CNN: best epoch / error | ResNet-18: best epoch / error |
|---|---:|---:|
| outer_01 | 7 / 5.3643° | 8 / 8.4464° |
| outer_02 | 6 / 6.7873° | 7 / 7.6621° |

The earlier two-fold development run used CUDA on an NVIDIA Tesla T4 (15 GiB reported VRAM). All four selected epochs match the minimum recorded error in their eight-epoch histories.

In [ ]:
# =============== 04. Train Fold-Specific CNN and ResNet Models ===============

# ---------------------- 1. Fit Inner Development Models ----------------------
history_rows,inner_metrics,outer_prediction_rows,checkpoint_rows=[],[],[],[]
for fold_i,fold_id in enumerate(active_folds):
    roles=outer_fold_indices(index,folds,fold_id)
    i_train,i_val,i_test=(roles[k] for k in ("train","validation","test"))
    print("Fold:",fold_id,"training people:",index.iloc[i_train].participant_id.nunique(),
          "inner Validation people:",index.iloc[i_val].participant_id.nunique())
    for model_name in MODEL_NAMES:
        pretrained=bool(model_name=="resnet18" and PRETRAINED_RESNET)
        network,history,choice=train_gaze_network(
            model_name,images,targets,i_train,i_val,device=device,
            epochs=EPOCHS,batch_size=BATCH_SIZE,
            seed=SEED+fold_i,pretrained=pretrained,
            learning_rate=(1e-4 if pretrained else LEARNING_RATE),
            patience=PATIENCE)
        for step in history:
            history_rows.append({"fold_id":fold_id,"model":model_name,**step})
        inner_metrics.append({"fold_id":fold_id,"model":model_name,
                              "n_fit_people":index.iloc[i_train].participant_id.nunique(),
                              "n_val_people":index.iloc[i_val].participant_id.nunique(),**choice})
        path=checkpoint_dir/f"04_{model_name}_{fold_id}.pt"
        torch.save({"state_dict":network.state_dict(),"fold_id":fold_id,
                    "model":model_name,"settings":choice},path)
        checkpoint_rows.append({"fold_id":fold_id,"model":model_name,
            "local_relative_path":path.relative_to(root).as_posix(),
            **choice})
        if RUN_OUTER_TEST:
            predicted=predict_network(network,images,i_test,device)
            part=index.iloc[i_test][["sample_id","participant_id"]].copy()
            part["fold_id"]=fold_id
            part["model"]=model_name
            part[["target_x","target_y","target_z"]]=targets[i_test]
            part[["pred_x","pred_y","pred_z"]]=predicted
            outer_prediction_rows.append(part)
        print(model_name,"| best epoch:",choice["best_epoch"],
              "| inner Validation angular error:",
              round(choice["inner_validation_mean_angular_error_deg"],3))

history_table=pd.DataFrame(history_rows)
inner_table=pd.DataFrame(inner_metrics)
checkpoints=pd.DataFrame(checkpoint_rows)
display(inner_table)

**05. Examine Inner Validation and Resource Requirements**

In the earlier development run, learning curves averaged the same two configurations at each epoch. The final plot includes a count of contributing folds at each epoch; later epoch means may represent fewer folds because of early stopping. All four earlier development fits reached eight epochs; none triggered early stopping. The Small CNN curve reaches its minimum at epoch seven and rises at epoch eight, demonstrating why a fold-specific best checkpoint is retained rather than the final epoch. The plotted epoch averages are distinct from the mean of the two individually selected checkpoint errors.

| Model | outer_01 (°) | outer_02 (°) | Mean of the two (°) |
|---|---:|---:|---:|
| Mean direction | 8.6726 | 9.1759 | 8.9243 |
| HOG + Ridge | 7.3446 | 7.7159 | 7.5303 |
| Small CNN | 5.3643 | 6.7873 | 6.0758 |
| ResNet-18 | 8.4464 | 7.6621 | 8.0542 |

The classical values are from the matching Notebook 03 fold configurations, not its 15-fold mean. Small CNN has the lowest selected error in both configurations and about 231 times fewer parameters than ResNet-18. Parameter count is not a measured runtime or memory comparison. The limited experiment does not isolate the effects of architecture, initialisation or learning rate.

In [ ]:
# =============== 05. Inspect CNN Learning and Inner Validation ===============

# ------------------------ 1. Summarise Observed Epochs ------------------------
epoch_summary = (history_table.groupby(["model", "epoch"])
                 .inner_validation_mean_angular_error_deg
                 .agg(mean_error="mean", n_folds="count").reset_index())
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True,
                         gridspec_kw={"height_ratios": [3, 1]})
for name, group in epoch_summary.groupby("model"):
    line, = axes[0].plot(group.epoch, group.mean_error, marker="o", label=name)
    axes[1].plot(group.epoch, group.n_folds, marker="o", color=line.get_color())
axes[0].set(ylabel="Inner Validation mean angular error (degrees)",
            title="Neural learning curves across observed fold epochs")
axes[0].legend()
axes[1].set(xlabel="Epoch", ylabel="Folds contributing")
axes[1].set_xticks(sorted(epoch_summary.epoch.unique()))
for ax in axes:
    ax.grid(alpha=0.2)
fig.tight_layout()
save_figure(fig, out_dir / "04_validation_curves.png")
plt.show()
print("Recorded model checkpoints stay under Git-ignored data/processed/")

**06. Save Model Evidence and Final Predictions**

The development run produced 32 history rows, four selected model/fold metric rows, four checkpoint metadata rows, four configuration rows and one learning-curve figure. The configuration records two evaluated folds and no Test-based selection. The checkpoint weights remain in the local processed-data directory.

In the current full outer-LOPO mode, both fixed model families must produce one held-out prediction per sample across all outer folds. Exact sample, participant, fold and target correspondence is checked before saving. Notebook 05 uses the saved prediction tables without training or selecting models.

In [ ]:
# =============== 06. Save Neural Development and Test Evidence ===============

# ------------------------ 1. Save Development Tables -------------------------
history_table.to_csv(out_dir / "04_training_history.csv", index=False)
inner_table.to_csv(out_dir / "04_inner_validation_metrics.csv", index=False)
checkpoints.to_csv(out_dir / "04_model_checkpoints.csv", index=False)

# ------------------------ 2. Save Model Configuration ------------------------
configuration = pd.DataFrame([
    {
        "fold_id": fold_id, "model": model_name,
        "run_mode": ("final_outer_lopo" if RUN_OUTER_TEST
                     else "development_inner_validation"),
        "all_outer_folds": len(all_folds),
        "evaluated_folds": len(active_folds),
        "imagenet_pretrained": bool(model_name == "resnet18" and PRETRAINED_RESNET),
        "epochs_max": EPOCHS, "patience": PATIENCE, "batch_size": BATCH_SIZE,
        "learning_rate": (1e-4 if model_name == "resnet18" and PRETRAINED_RESNET
                          else LEARNING_RATE),
        "seed": SEED + fold_i, "test_used_for_model_selection": False,
    }
    for fold_i, fold_id in enumerate(active_folds)
    for model_name in MODEL_NAMES
])
configuration.to_csv(out_dir / "04_run_configuration.csv", index=False)

# ------------------------ 3. Save Outer Test Evidence ------------------------
if RUN_OUTER_TEST:
    predictions=pd.concat(outer_prediction_rows,ignore_index=True)
    predictions, person_metrics = reconcile_outer_predictions(
        predictions, index, folds, MODEL_NAMES)
    predictions.to_csv(out_dir/"04_neural_outer_predictions.csv",index=False)
    person_metrics.to_csv(out_dir/"04_neural_outer_metrics.csv",index=False)
    print("Final outer-person predictions:",len(predictions))
else:
    print("Outer Test not scored; no Test prediction table written")

**07. Interpretation and Output Download**

The two-fold development run establishes that both neural training procedures execute and select their checkpoints using inner Validation. Small CNN shows lower error than the three matching comparators in these configurations, but the evidence is preliminary. The full outer-LOPO run remains necessary for the participant-level evaluation.

The final run exports the stage tables, plot and full outer predictions; raw image arrays and checkpoint weights remain local. Notebook 05 requires the complete outer prediction tables from Notebooks 03 and 04.

In [ ]:
# ================== 07. Interpret Neural Results and Export ==================

needed=["04_training_history.csv","04_inner_validation_metrics.csv",
 "04_model_checkpoints.csv","04_run_configuration.csv",
 "04_validation_curves.png"]
if RUN_OUTER_TEST:
    needed+=["04_neural_outer_predictions.csv","04_neural_outer_metrics.csv"]
missing=[x for x in needed if not (out_dir/x).is_file()]
if missing:
    raise FileNotFoundError(f"Notebook 04 stage files missing: {missing}")
print("Recorded inner Validation model/fold configurations:",len(inner_table))
print("No new Test comparison has been used for model selection")
print("Next: 05_cross_person_evaluation_and_robustness.ipynb")
(out_dir / "04_run_configuration.json").unlink(missing_ok=True)
download_outputs_from_colab("vision_based_gaze_estimation/output","04")